In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3"
  },
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# ARIA — Autonomous Research & Iteration Agent\n",
    "## Training Notebook\n",
    "**Author:** Angel Singh\n",
    "**Hackathon:** Meta PyTorch OpenEnv Hackathon × Scaler 2026\n",
    "\n",
    "This notebook trains an LLM agent to autonomously complete enterprise workflows using GRPO via HuggingFace TRL + Unsloth."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 1 — Install Dependencies"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!pip install openenv unsloth trl transformers gradio fastapi uvicorn matplotlib -q\n",
    "print('✅ All dependencies installed!')"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 2 — Clone Repository"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!git clone https://github.com/yourusername/aria-env\n",
    "%cd aria-env\n",
    "print('✅ Repository cloned!')"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 3 — Verify Environment"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "from environment.aria_env import ARIAEnvironment\n",
    "\n",
    "env = ARIAEnvironment(capped=True, difficulty=1)\n",
    "obs = env.reset()\n",
    "print('✅ Environment working!')\n",
    "print('Step:', obs['step'])\n",
    "print('Tasks:', obs['tasks_completed'], '/', obs['total_tasks'])\n",
    "env.render()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 4 — Test Reward Model"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "from environment.reward import RewardModel\n",
    "\n",
    "reward_model = RewardModel(capped=True)\n",
    "reward = reward_model.compute(\n",
    "    tasks_completed=5,\n",
    "    total_tasks=5,\n",
    "    tool_calls=20,\n",
    "    min_tool_calls=20,\n",
    "    adaptation_triggered=True,\n",
    "    policy_changed=True,\n",
    "    action_history=[],\n",
    ")\n",
    "print('✅ Reward Model working!')\n",
    "print('Reward:', reward)\n",
    "print('Breakdown:', reward_model.get_last_reward_breakdown())"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 5 — Run Baseline Demo"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!python demo.py"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 6 — Load Model with Unsloth"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "from unsloth import FastLanguageModel\n",
    "import torch\n",
    "\n",
    "model, tokenizer = FastLanguageModel.from_pretrained(\n",
    "    model_name='Qwen/Qwen2.5-7B-Instruct',\n",
    "    max_seq_length=1024,\n",
    "    load_in_4bit=True,\n",
    "    fast_inference=True,\n",
    ")\n",
    "print('✅ Model loaded!')\n",
    "print('Model:', model.__class__.__name__)"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 7 — Define Reward Function for GRPO"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "from environment.aria_env import ARIAEnvironment\n",
    "from training.train import build_prompt, parse_action\n",
    "\n",
    "def aria_reward_function(prompts, completions, **kwargs):\n",
    "    rewards = []\n",
    "    for completion in completions:\n",
    "        env = ARIAEnvironment(capped=True, difficulty=1)\n",
    "        obs = env.reset()\n",
    "        total_reward = 0.0\n",
    "        for _ in range(20):\n",
    "            action = parse_action(completion)\n",
    "            obs, reward, done, info = env.step(action)\n",
    "            total_reward += reward\n",
    "            if done:\n",
    "                break\n",
    "        rewards.append(total_reward)\n",
    "    return rewards\n",
    "\n",
    "print('✅ Reward function defined!')"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 8 — Train with GRPO"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "from trl import GRPOTrainer, GRPOConfig\n",
    "from environment.aria_env import ARIAEnvironment\n",
    "from training.train import build_prompt\n",
    "\n",
    "# Build training prompts\n",
    "env = ARIAEnvironment(capped=True, difficulty=1)\n",
    "obs = env.reset()\n",
    "prompt = build_prompt(obs)\n",
    "\n",
    "train_dataset = [{\"prompt\": prompt} for _ in range(50)]\n",
    "\n",
    "# GRPO Config\n",
    "grpo_config = GRPOConfig(\n",
    "    learning_rate=5e-6,\n",
    "    per_device_train_batch_size=1,\n",
    "    gradient_accumulation_steps=4,\n",
    "    num_generations=4,\n",
    "    max_prompt_length=512,\n",
    "    max_completion_length=256,\n",
    "    num_train_epochs=1,\n",
    "    logging_steps=5,\n",
    "    output_dir='./results',\n",
    "    report_to='none',\n",
    ")\n",
    "\n",
    "# Trainer\n",
    "trainer = GRPOTrainer(\n",
    "    model=model,\n",
    "    tokenizer=tokenizer,\n",
    "    reward_funcs=aria_reward_function,\n",
    "    args=grpo_config,\n",
    "    train_dataset=train_dataset,\n",
    ")\n",
    "\n",
    "print('✅ Trainer ready! Starting training...')\n",
    "trainer.train()\n",
    "print('✅ Training complete!')"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 9 — Generate Reward Curves"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!python demo.py\n",
    "print('✅ Graphs saved to results/')"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 10 — Save Model to HuggingFace"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "from huggingface_hub import login\n",
    "login(token='your_token_here')\n",
    "\n",
    "model.save_pretrained('./aria-trained-model')\n",
    "tokenizer.save_pretrained('./aria-trained-model')\n",
    "\n",
    "model.push_to_hub('angel-singh/aria-trained-model')\n",
    "tokenizer.push_to_hub('angel-singh/aria-trained-model')\n",
    "print('✅ Model pushed to HuggingFace!')"
   ],
   "outputs": [],
   "execution_count": null
  }
 ]
}